# FAIRyMAGs Use Case Analysis

This notebook analyzes MAGs quality across four use cases:
- **Bee**: Honeybee gut microbiome
- **Cloud**: Cloud microbiome
- **Marine**: Marine microbiome
- **Termite**: Termite gut microbiome

## Data
Input data is stored in `../data/{use-case}/data/`:
- `drep.csv`: dRep clustering output
- `checkm2.tsv`: CheckM2 quality assessment
- `gtdb.tsv`: GTDB taxonomy classification
- `quast.tsv`: QUAST assembly statistics
- `bakta.tsv`: BAKTA genome annotation

## Output
- Plots: `../results/{use-case}/`
- Summary tables: `../results/`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pathlib

# === CONFIGURATION ===
DATA_DIR = pathlib.Path("../data")
RESULTS_DIR = pathlib.Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

USE_CASES = ["bee-use-case", "cloud-use-case", "marine-use-case", "termite-use-case"]
DISPLAY_NAMES = {
    "bee-use-case": "Bee",
    "cloud-use-case": "Cloud",
    "marine-use-case": "Marine",
    "termite-use-case": "Termite",
}

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Results directory: {RESULTS_DIR.resolve()}")

## Part 1: MAGs Quality Distribution

In [ ]:
def get_data_path(use_case):
    return DATA_DIR / use_case

def count_drep_bins(filename):
    count = 0
    with open(filename, "r") as f:
        for i, line in enumerate(f):
            if i == 0 or not line.strip():
                continue
            count += 1
    return count

def assign_threshold(comp):
    if comp >= 90:
        return ">90"
    elif comp >= 80:
        return ">80"
    elif comp >= 70:
        return ">70"
    elif comp >= 60:
        return ">60"
    elif comp >= 50:
        return ">50"
    return None

thresholds = [">90", ">80", ">70", ">60", ">50"]
colors = ["#004c4c", "#006666", "#008080", "#33a3a3", "#99d6d6"]

def load_counts(cont_cutoff):
    result = {thr: [] for thr in thresholds}
    totals = []

    for usecase in USE_CASES:
        f = get_data_path(usecase) / "checkm2.tsv"
        df = pd.read_csv(f, sep="\t")
        df2 = df[df["Contamination"] < cont_cutoff].copy()
        df2["Thr"] = df2["Completeness"].apply(assign_threshold)

        counts = df2["Thr"].value_counts()

        total = 0
        for thr in thresholds:
            val = counts.get(thr, 0)
            result[thr].append(val)
            total += val

        totals.append(total)

    usecases = [DISPLAY_NAMES[uc] for uc in USE_CASES]
    sorting_idx = np.argsort(totals)[::-1]

    sorted_use_cases = [usecases[i] for i in sorting_idx]
    sorted_result = {
        thr: [result[thr][i] for i in sorting_idx]
        for thr in thresholds
    }

    return sorted_use_cases, sorted_result

def plot_stacked(ax, tools, data, title):
    y = np.arange(len(tools))
    left = np.zeros(len(tools))

    for i, thr in enumerate(thresholds):
        values = data[thr]
        ax.barh(
            y,
            values,
            left=left,
            color=colors[i],
            label=thr,
            height=0.6
        )
        left += np.array(values)

    for i in range(len(tools)):
        ax.text(
            left[i] + (max(left) * 0.02),
            i,
            str(int(left[i])),
            va="center",
            ha="left",
            fontsize=15
        )

    ax.set_xlim(0, max(left) * 1.25)
    ax.set_yticks(y)
    ax.set_yticklabels(tools, fontsize=15)
    ax.set_xlabel("#MAGs", fontsize=15)
    ax.set_title(title, fontsize=18)
    ax.tick_params(axis="x", labelsize=15)

# Load data
use_cases, data5 = load_counts(cont_cutoff=5)
_, data10 = load_counts(cont_cutoff=10)

# Side-by-side plots
fig, axs = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

plot_stacked(axs[0], use_cases, data5, "Contamination < 5%")
plot_stacked(axs[1], use_cases, data10, "Contamination < 10%")

# Panel labels
axs[0].text(
    -0.12, 1.05, "(A)",
    transform=axs[0].transAxes,
    fontsize=20,
    fontweight="bold",
    va="top",
    ha="left"
)

axs[1].text(
    -0.12, 1.05, "(B)",
    transform=axs[1].transAxes,
    fontsize=20,
    fontweight="bold",
    va="top",
    ha="left"
)

# Legend
axs[1].legend(
    title="Completeness",
    loc="upper right",
    fontsize=15,
    title_fontsize=16
)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "combined_bar_plot.png", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "combined_bar_plot.svg", dpi=300, bbox_inches="tight")
plt.show()

# Print dRep counts
for uc in USE_CASES:
    drep_file = get_data_path(uc) / "drep.csv"
    print(f"{DISPLAY_NAMES[uc]}: dRep found {count_drep_bins(drep_file)} MAGs")

## Part 2: Summary Table Generation

In [ ]:
def extract_rank(classification, rank):
    """Extract a GTDB rank from a classification string."""
    prefix_map = {
        "domain": "d__", "phylum": "p__", "class": "c__",
        "order": "o__", "family": "f__", "genus": "g__", "species": "s__"
    }
    prefix = prefix_map.get(rank)
    if pd.isna(classification) or not prefix:
        return None
    for part in str(classification).split(";"):
        part = part.strip()
        if part.startswith(prefix):
            name = part[len(prefix):].strip()
            return name if name else "unclassified"
    return "unclassified"

def load_transposed_tsv(filepath):
    """Load a TSV where rows=metrics, columns=samples."""
    df = pd.read_csv(filepath, sep="\t", index_col=0).T
    df.index = df.index.str.replace(r"\.fasta$", "", regex=True)
    return df

def load_transposed_bakta(filepath):
    """Load bakta TSV where rows=annotations, columns=samples."""
    df = pd.read_csv(filepath, sep="\t", index_col=0)
    df = df.T
    df.index = df.index.str.replace(r"\.fasta_2$", "", regex=True)
    df.index = df.index.str.replace(r"\.fasta$", "", regex=True)
    return df

summary_rows = []
per_ucase_tables = {}

for uc in USE_CASES:
    dat = get_data_path(uc)
    uc_name = DISPLAY_NAMES[uc]

    # --- dRep ---
    drep = pd.read_csv(dat / "drep.csv")
    drep["genome"] = drep["genome"].str.replace(r"\.fasta$", "", regex=True)
    total_mags = len(drep)
    n_clusters = drep["secondary_cluster"].nunique()
    cluster_sizes = drep.groupby("secondary_cluster").size()
    multi_member = (cluster_sizes >= 2).sum()

    # --- CheckM2 & GTDB (cluster reps) ---
    checkm = pd.read_csv(dat / "checkm2.tsv", sep="\t")
    checkm["Name"] = checkm["Name"].str.replace(r"\.fasta$", "", regex=True)
    gtdb = pd.read_csv(dat / "gtdb.tsv", sep="\t")
    gtdb["user_genome"] = gtdb["user_genome"].str.replace(r"\.fasta$", "", regex=True)
    merged = pd.merge(checkm, gtdb, left_on="Name", right_on="user_genome", how="left")
    merged["phylum"] = merged["classification"].apply(lambda x: extract_rank(x, "phylum"))
    merged["genus"] = merged["classification"].apply(lambda x: extract_rank(x, "genus"))
    merged["species"] = merged["classification"].apply(lambda x: extract_rank(x, "species"))
    merged.loc[merged["species"].isna(), "species"] = "no GTDB hit"

    # --- Quality tiers (MIMAG) ---
    n_near_complete = ((merged["Completeness"] > 95) & (merged["Contamination"] < 5)).sum()
    n_high = ((merged["Completeness"] > 90) & (merged["Contamination"] < 5)).sum()
    n_medium = ((merged["Completeness"] >= 50) & (merged["Contamination"] < 10)).sum()
    n_low = len(merged) - n_medium
    n_medium_only = n_medium - n_high  # medium but not high

    # --- Robust completeness/contamination stats ---
    comp_med = merged["Completeness"].median()
    comp_q1 = merged["Completeness"].quantile(0.25)
    comp_q3 = merged["Completeness"].quantile(0.75)
    cont_med = merged["Contamination"].median()
    cont_q1 = merged["Contamination"].quantile(0.25)
    cont_q3 = merged["Contamination"].quantile(0.75)

    # --- Unclassified stats ---
    species_counts = merged["species"].value_counts()
    n_unclassified = species_counts.get("unclassified", 0)
    n_no_gtdb = species_counts.get("no GTDB hit", 0)

    # --- Top 30 clusters by size (with taxonomy) ---
    def tax_label(classification):
        """Return lowest resolved rank with GTDB prefix if not species-level."""
        if pd.isna(classification):
            return "no GTDB hit"
        ranks = [("s__", None), ("g__", "g__"), ("f__", "f__"), ("p__", "p__")]
        for prefix, label in ranks:
            for part in str(classification).split(";"):
                part = part.strip()
                if part.startswith(prefix):
                    name = part[len(prefix):].strip()
                    if name:
                        return name if label is None else f"{label}{name}"
        return "no GTDB hit"

    drep_merged = pd.merge(drep, gtdb[["user_genome", "classification"]],
                           left_on="genome", right_on="user_genome", how="left")
    cluster_tax = drep_merged.groupby("secondary_cluster").agg(
        size=("genome", "size"),
        classification=("classification", "first"),
    ).sort_values("size", ascending=False)
    cluster_tax["label"] = cluster_tax["classification"].apply(tax_label)
    cluster_tax = cluster_tax[cluster_tax["label"] != "no GTDB hit"]
    top30_taxa = "; ".join(
        f"{row['label']} ({row['size']})"
        for _, row in cluster_tax.head(30).iterrows()
    )

    summary_rows.append({
        "Use case": uc_name,
        "Total MAGs": total_mags,
        "Species-level clusters": n_clusters,
        "Clusters >=2": multi_member,
        "Avg completeness": f"{merged['Completeness'].mean():.1f}",
        "Completeness range": f"{merged['Completeness'].min():.1f}–{merged['Completeness'].max():.1f}",
        "Avg contamination": f"{merged['Contamination'].mean():.2f}",
        "Contamination range": f"{merged['Contamination'].min():.2f}–{merged['Contamination'].max():.2f}",
        "Top 30 taxa": top30_taxa,
    })

    # --- Per-use-case all cluster reps ---
    reps = merged.sort_values("Completeness", ascending=False).copy()
    cluster_size_map = cluster_sizes.to_dict()
    reps["Cluster members"] = reps["Name"].map(
        lambda n: cluster_size_map.get(
            drep.loc[drep["genome"] == n, "secondary_cluster"].iloc[0], 0
        ) if n in drep["genome"].values else 0
    )
    
    # GTDB 7 ranks
    for rank in ["domain", "phylum", "class", "order", "family", "genus", "species"]:
        reps[rank] = reps["classification"].apply(lambda x: extract_rank(x, rank))
    reps.loc[reps["species"].isna(), "species"] = "no GTDB hit"
    
    # QUAST stats
    quast = load_transposed_tsv(dat / "quast.tsv")
    quast_cols = [c for c in quast.columns if c not in ("Assembly",)]
    reps = pd.merge(reps, quast[quast_cols], left_on="Name", right_index=True, how="left")
    
    # Bakta stats
    bakta = load_transposed_bakta(dat / "bakta.tsv")
    bakta_cols = [c for c in bakta.columns if c not in ("Annotation", "Count")]
    bakta = bakta[bakta_cols].add_prefix("bakta_")
    reps = pd.merge(reps, bakta, left_on="Name", right_index=True, how="left")
    
    # CheckM (v1) stats
    checkm_v1 = pd.read_csv(dat / "checkm.tsv", sep="\t")
    checkm_v1["Bin Id"] = checkm_v1["Bin Id"].str.replace(r"\.fasta$", "", regex=True)
    checkm_v1 = checkm_v1.set_index("Bin Id")
    checkm_v1_cols = ["Strain heterogeneity", "# markers", "# marker sets"]
    reps = pd.merge(reps, checkm_v1[checkm_v1_cols], left_on="Name", right_index=True, how="left")
    
    # CoverM stats (mean coverage across samples)
    coverm = pd.read_csv(dat / "coverm.tsv", sep="\t")
    coverm["Genome"] = coverm["Genome"].str.replace(r"\.fasta$", "", regex=True)
    coverm_mean = coverm.set_index("Genome").mean(axis=1).rename("coverm_mean_coverage")
    reps = pd.merge(reps, coverm_mean, left_on="Name", right_index=True, how="left")
    
    # KEGG pathway completeness (wide: one column per pathway)
    kegg = pd.read_csv(dat / "kegg_pathway_completeness.tsv", sep="\t")
    kegg["contig"] = kegg["contig"].str.replace(r"\.fasta$", "", regex=True)
    kegg_pivot = kegg.pivot_table(
        index="contig", columns="pathway_name", values="completeness", aggfunc="max"
    ).add_prefix("kegg_")
    reps = pd.merge(reps, kegg_pivot, left_on="Name", right_index=True, how="left")
    
    # Final column selection
    reps["Completeness"] = reps["Completeness"].round(1).astype(str) + "%"
    reps["Contamination"] = reps["Contamination"].round(2).astype(str) + "%"
    
    # Rename base columns
    reps = reps.rename(columns={
        "Name": "MAG", "domain": "Domain", "phylum": "Phylum",
        "class": "Class", "order": "Order", "family": "Family",
        "genus": "Genus", "species": "Species"
    })
    
    # Select all columns in order
    all_cols = ["MAG", "Domain", "Phylum", "Class", "Order", "Family", "Genus", "Species",
                "Cluster members", "Completeness", "Contamination"]
    all_cols += quast_cols
    all_cols += [c for c in reps.columns if c.startswith("bakta_")]
    all_cols += checkm_v1_cols
    all_cols += ["coverm_mean_coverage"]
    all_cols += [c for c in reps.columns if c.startswith("kegg_")]
    reps = reps[all_cols]
    per_ucase_tables[uc_name] = reps.reset_index(drop=True)

# --- Assemble & save summary ---
summary = pd.DataFrame(summary_rows)
summary_cols = [
    "Use case", "Total MAGs", "Species-level clusters", "Clusters >=2",
    "Avg completeness", "Completeness range",
    "Avg contamination", "Contamination range",
    "Top 30 taxa",
]
summary = summary[summary_cols]

summary.to_csv(RESULTS_DIR / "summary.tsv", sep="\t", index=False)
print(f"Saved: {RESULTS_DIR / 'summary.tsv'}")

# --- Table: Top 30 taxa + All phyla ---
top30_only = summary[["Use case", "Top 30 taxa"]].copy()

# --- All phyla per use case (total MAGs per phylum) ---
phyla_rows = []
for uc in USE_CASES:
    dat = get_data_path(uc)
    uc_name = DISPLAY_NAMES[uc]
    drep = pd.read_csv(dat / "drep.csv")
    drep["genome"] = drep["genome"].str.replace(r"\.fasta$", "", regex=True)
    cluster_sizes_all = drep.groupby("secondary_cluster").size().to_dict()
    gtdb = pd.read_csv(dat / "gtdb.tsv", sep="\t")
    gtdb["user_genome"] = gtdb["user_genome"].str.replace(r"\.fasta$", "", regex=True)
    gtdb["phylum"] = gtdb["classification"].apply(lambda x: extract_rank(x, "phylum"))
    gtdb_phyl = gtdb[gtdb["phylum"].notna() & (gtdb["phylum"] != "unclassified")][["user_genome", "phylum"]]
    drep_gtdb = pd.merge(drep, gtdb_phyl, left_on="genome", right_on="user_genome", how="inner")
    cluster_phyla = drep_gtdb.drop_duplicates("secondary_cluster")[["secondary_cluster", "phylum"]]
    cluster_phyla["cluster_size"] = cluster_phyla["secondary_cluster"].map(cluster_sizes_all)
    counts = cluster_phyla.groupby("phylum")["cluster_size"].sum().sort_values(ascending=False)
    all_phyla = "; ".join(f"{p} ({c})" for p, c in counts.items())
    phyla_rows.append({"Use case": uc_name, "All phyla": all_phyla})
phyla_df = pd.DataFrame(phyla_rows)

taxa_phyla = pd.merge(top30_only, phyla_df, on="Use case")
taxa_phyla.to_csv(RESULTS_DIR / "taxa_phyla.tsv", sep="\t", index=False)
print(f"Saved: {RESULTS_DIR / 'taxa_phyla.tsv'}")

summary

In [ ]:
# All cluster reps per use case (sorted by completeness)
for uc in USE_CASES:
    uc_name = DISPLAY_NAMES[uc]
    tbl = per_ucase_tables[uc_name]
    print(f"\n=== {uc_name} ===")
    print(tbl.to_string(index=False))
    out_dir = RESULTS_DIR / uc
    out_dir.mkdir(exist_ok=True)
    out_file = out_dir / f"reps_{uc_name.lower()}.tsv"
    tbl.to_csv(out_file, sep="\t", index=False)
    print(f"Saved: {out_file}")

## Discussion: completeness & contamination — metrics and improvement

### Better representation of quality
- **Median + IQR** (shown above) is more robust than mean for skewed distributions.
- **MIMAG quality tiers** (near-complete/high/medium/low) provide actionable thresholds.
- The Cloud use case is dominated by low-quality MAGs (median completeness ~56%). Showing only
  mean/min/max hides the bimodal nature — a few high-quality reps pull the max to 100%.

### How to improve completeness
- **Deeper sequencing** — especially for Cloud (low biomass, high strain diversity).
- **Co-assembly** of related samples boosts contiguity for shared community members.
- **Iterative binning** (MetaBAT2 + CONCOCT + MaxBin → DAS_Tool / MetaWRAP) recovers more
  genomes per sample.
- **Long reads** (Oxford Nanopore / PacBio HiFi) close fragmented bins.
- **Differential coverage** across multiple samples is the single strongest signal for binning.

### How to reduce contamination
- **RefineM / MAGIC / GUNC** identify chimeric bins by GC, coverage, and taxonomic marker
  consistency.
- **Manual curation** via anvi'o or VizBin for high-priority clusters (e.g., novel species).
- **Tighter binning parameters** (e.g., MetaBAT2 `--minCV` for stricter coverage variance).
- The Cloud use-case shows several reps with 10–25% contamination despite low completeness —
  suggesting chimeric bins from diverse strain populations.
- **Single-copy marker gene counts** (CheckM/CheckM2) flag multi-contamination early.